In [1]:
import pandas as pd
import numpy as np
import os

# 1. Safely locate data path
base_dir = os.getcwd()
data_path = os.path.join(base_dir, 'data', 'online_retail.csv')

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# 2. Inspect the raw state
print("\n--- Raw Data Shape ---")
print(df.shape)

# 3. Handle missing critical identifiers safely
# We cannot run a behavioral cohort analysis on anonymous traffic
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)

# 4. Filter out technical errors & order cancellations 
# Cancellations are marked with a 'C' prefix in InvoiceNo; Quantity/Price should be positive
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 5. Calculate direct revenue metric
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']

# 6. Parse date structure safely
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

print("\n--- Cleaned Data Shape ---")
print(df.shape)
df.head()

Loading dataset from: c:\Users\DELL\OneDrive\Desktop\Business_ Analytics_Project\data\online_retail.csv

--- Raw Data Shape ---
(541909, 8)

--- Cleaned Data Shape ---
(397884, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalRevenue,InvoiceMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


In [12]:
# 1. Isolate the 'Birth Month' for every customer profile
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

# 2. Compute aggregate unique user matrix grouped by cohort cohorts
cohort_data = df.groupby(['CohortMonth', 'InvoiceMonth']).agg({'CustomerID': 'nunique'}).reset_index()

# 3. Extract the integer index representing months since onboarding
cohort_data['MonthIndex'] = (cohort_data['InvoiceMonth'].dt.year - cohort_data['CohortMonth'].dt.year) * 12 + \
                            (cohort_data['InvoiceMonth'].dt.month - cohort_data['CohortMonth'].dt.month)

# 4. Pivot into a visual matrix form
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='MonthIndex', values='CustomerID')

# 5. Transform absolute user counts into precise percentage retention metrics
cohort_sizes = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0)

print("\n--- Retention Matrix Built Successfully ---")
print("=" * 70 )
print(retention_matrix.round(3) * 100)


--- Retention Matrix Built Successfully ---
MonthIndex      0     1     2     3     4     5     6     7     8     9   \
CohortMonth                                                                
2010-12      100.0  36.6  32.3  38.4  36.3  39.8  36.3  34.9  35.4  39.5   
2011-01      100.0  22.1  26.6  23.0  32.1  28.8  24.7  24.2  30.0  32.6   
2011-02      100.0  18.7  18.7  28.4  27.1  24.7  25.3  27.9  24.7  30.5   
2011-03      100.0  15.0  25.2  19.9  22.3  16.8  26.8  23.0  27.9   8.6   
2011-04      100.0  21.3  20.3  21.0  19.7  22.7  21.7  26.0   7.3   NaN   
2011-05      100.0  19.0  17.3  17.3  20.8  23.2  26.4   9.5   NaN   NaN   
2011-06      100.0  17.4  15.7  26.4  23.1  33.5   9.5   NaN   NaN   NaN   
2011-07      100.0  18.1  20.7  22.3  27.1  11.2   NaN   NaN   NaN   NaN   
2011-08      100.0  20.7  24.9  24.3  12.4   NaN   NaN   NaN   NaN   NaN   
2011-09      100.0  23.4  30.1  11.4   NaN   NaN   NaN   NaN   NaN   NaN   
2011-10      100.0  24.0  11.5   NaN   NaN 

In [11]:
# We establish a baseline tracking date right after the latest transaction
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# 1. Aggregate core metrics per customer profile
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalRevenue': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalRevenue': 'Monetary'
})

# 2. Divide customers into performance tiers using quantiles
r_labels = range(4, 0, -1)  # Low recency number = bad (haven't seen them in a long time)
f_labels = range(1, 5)     # High frequency number = great (buy constantly)
m_labels = range(1, 5)     # High monetary number = great (big spenders)

rfm['R'] = pd.qcut(rfm['Recency'], q=4, labels=r_labels)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=f_labels)
rfm['M'] = pd.qcut(rfm['Monetary'], q=4, labels=m_labels)

# 3. Concatenate scores into a unified behavioral identifier string
rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

# 4. Map numerical combinations to concrete operational business tiers
def assign_tier(df_row):
    segment = df_row['RFM_Segment']
    if segment in ['444', '443', '434']:
        return 'Champions / VIP'
    elif segment.startswith('4') or segment.startswith('3'):
        return 'Loyal & Active'
    elif segment.startswith('2'):
        return 'At Risk / Slipping'
    else:
        return 'Hibernating / Lost'

rfm['Business_Segment'] = rfm.apply(assign_tier, axis=1)

# 5. Export clean files out for downstream BI Dashboarding tool ingestion
os.makedirs('data_output', exist_ok=True)
rfm.to_csv('data_output/customer_rfm_segments.csv')
print("\n--- RFM Segment Export finalized ---")
print("=" * 35 )
print(rfm['Business_Segment'].value_counts())


--- RFM Segment Export finalized ---
Business_Segment
Loyal & Active        1524
Hibernating / Lost    1084
At Risk / Slipping    1066
Champions / VIP        664
Name: count, dtype: int64
